# Manual review: resolving flagged identity-claim roles

Walks through `flagged_for_review.csv` (from the identity-claim runner) one
row at a time. For each row you can:

- type a role name (e.g. `Seer`), or multiple comma-separated if the line
  genuinely claims more than one (e.g. `Villager, Robber`) → resolves the
  claim and writes it directly into the annotation JSON
- press Enter (empty input) or type `u` → confirms the role is genuinely
  unresolvable; stays `["UNKNOWN"]`, but is marked as manually reviewed so
  you know it's been looked at, not just skipped
- type `s` → skip this row for now (comes back next time you run the loop)
- type `q` → stop the loop early; everything done so far is already saved

Have the source transcript open in another window
(`identity_claim_transcripts/ready_for_annotation/<source>/<...>.txt`) — use
the line number printed for each row to jump to it.

Progress is saved after every single answer (both to the CSV and to the
annotation JSON), so it's safe to stop and resume anytime.


In [1]:
import json
import csv
import os
from pathlib import Path

# ---- adjust these if your paths differ ----
CSV_PATH = Path(
    r"C:\Users\annab\Documents\GitHub\masters_thesis_sdg\data\processed"
    r"\lai2023\identity_claim_transcripts\ic_targets\flagged_for_review.csv"
)
OUTPUT_ROOT = Path(
    r"C:\Users\annab\Documents\GitHub\masters_thesis_sdg\data\processed"
    r"\lai2023\identity_claim_transcripts\ic_targets"
)
# --------------------------------------------

FIELDNAMES_EXTRA = ["review_status", "resolved_roles"]


def load_rows():
    with CSV_PATH.open(newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        rows = list(reader)
    # add tracking columns if this is the first time we're running the notebook
    for row in rows:
        row.setdefault("review_status", "")       # "", "resolved", "confirmed_unknown", "skipped"
        row.setdefault("resolved_roles", "")
    return rows


def save_rows(rows):
    fieldnames = list(rows[0].keys()) if rows else []
    with CSV_PATH.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)


def update_annotation_json(row, new_roles):
    """Find the matching item inside the game's output JSON and update it.
    new_roles: list[str] to resolve to, or None to just mark as manually
    confirmed while leaving claimed_roles=["UNKNOWN"] as is.
    """
    json_path = OUTPUT_ROOT / row["output_file"].replace(chr(92), os.sep).replace("/", os.sep)
    record = json.loads(json_path.read_text(encoding="utf-8"))

    target_line = int(row["line_number"])
    target_evidence = row["evidence"]

    for item in record.get("items", []):
        if item.get("line_number") != target_line:
            continue
        if item.get("claimed_roles") != ["UNKNOWN"]:
            continue
        # evidence as a tiebreaker in the rare case of duplicate UNKNOWN claims on one line
        if item.get("evidence") != target_evidence:
            continue

        item["manually_reviewed"] = True
        if new_roles is not None:
            item["claimed_roles"] = new_roles
            item["requires_review"] = False
        else:
            item["confirmed_unresolvable"] = True
            # requires_review stays True -- it's still an UNKNOWN, just a confirmed one

        json_path.write_text(
            json.dumps(record, indent=2, ensure_ascii=False), encoding="utf-8"
        )
        return True

    print(f"  WARNING: could not find matching item in {json_path} "
          f"(line {target_line}) -- nothing was updated.")
    return False


rows = load_rows()
pending = [r for r in rows if r["review_status"] not in ("resolved", "confirmed_unknown")]
print(f"{len(rows)} total flagged rows, {len(pending)} still pending review.")


8 total flagged rows, 8 still pending review.


In [2]:
for row in pending:
    print("=" * 70)
    print(f"File     : {row['output_file']}")
    print(f"Game     : {row['game']}   (session: {row['session']})")
    print(f"Line     : {row['line_number']}")
    print(f"Evidence : {row['evidence']}")
    print("-" * 70)

    answer = input(
        "Role(s), comma-separated / Enter or 'u' = confirm unresolvable / "
        "'s' = skip / 'q' = quit: "
    ).strip()

    if answer.lower() == "q":
        print("Stopping. Progress so far is saved.")
        break

    if answer.lower() == "s":
        row["review_status"] = "skipped"
        save_rows(rows)
        continue

    if answer == "" or answer.lower() == "u":
        update_annotation_json(row, new_roles=None)
        row["review_status"] = "confirmed_unknown"
        save_rows(rows)
        print("-> confirmed unresolvable.")
        continue

    roles = [r.strip() for r in answer.split(",") if r.strip()]
    ok = update_annotation_json(row, new_roles=roles)
    if ok:
        row["review_status"] = "resolved"
        row["resolved_roles"] = ", ".join(roles)
        save_rows(rows)
        print(f"-> resolved to {roles}.")
    else:
        print("-> NOT saved (see warning above). Row left pending -- try again.")

print("=" * 70)
remaining = [r for r in rows if r["review_status"] not in ("resolved", "confirmed_unknown")]
print(f"Done for now. {len(remaining)} rows still pending "
      f"(skipped ones will show up again next run).")


File     : Youtube\ONE#NIGHT#ULTIMATE#WEREWOLF#-#Drinking#Play#Through_Game1.json
Game     : Game1   (session: ONE#NIGHT#ULTIMATE#WEREWOLF#-#Drinking#Play#Through)
Line     : 38
Evidence : I was.
----------------------------------------------------------------------
-> resolved to ['troublemaker'].
File     : Youtube\ONE#NIGHT#ULTIMATE#WEREWOLF#-#Drinking#Play#Through_Game1.json
Game     : Game1   (session: ONE#NIGHT#ULTIMATE#WEREWOLF#-#Drinking#Play#Through)
Line     : 106
Evidence : I was.
----------------------------------------------------------------------
-> resolved to ['hunter'].
File     : Youtube\ONE#NIGHT#ULTIMATE#WEREWOLF#-#Drinking#Play#Through_Game1.json
Game     : Game1   (session: ONE#NIGHT#ULTIMATE#WEREWOLF#-#Drinking#Play#Through)
Line     : 119
Evidence : Yes.
----------------------------------------------------------------------
-> resolved to ['hunter'].
File     : Youtube\ONE#NIGHT#ULTIMATE#WEREWOLF#-#Drinking#Play#Through_Game2.json
Game     : Game2   (session: O

Stopped partway, or came back later? Re-run the config cell above (it reloads the CSV with your saved progress), then run the cell below to rebuild `pending`, then re-run the loop cell.

In [3]:
# Re-run this cell (instead of the one above) to only go through rows you
# explicitly skipped last time, without re-listing already-resolved ones.
pending = [r for r in rows if r["review_status"] not in ("resolved", "confirmed_unknown")]
print(f"{len(pending)} rows pending (including previously skipped).")


0 rows pending (including previously skipped).
